# NOAA NDVI → VirtualiZarr → Local Icechunk

Demonstrates creating virtual references to public NOAA AVHRR NDVI NetCDF files
on S3 and writing them to a local icechunk repository — no data copying required.

We use a small subset (6 time steps) to test the workflow.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import icechunk
import xarray as xr
import s3fs
from obstore.store import S3Store
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser

In [ ]:
# Public NOAA NDVI bucket — no credentials needed
BUCKET = 'noaa-cdr-ndvi-pds'
PREFIX = 'data/2026'
N_FILES = 6  # number of time steps to test

fs = s3fs.S3FileSystem(anon=True)
all_files = sorted(fs.ls(f'{BUCKET}/{PREFIX}'))
files = all_files[:N_FILES]
urls = [f's3://{f}' for f in files]
print(f'Using {len(urls)} files:')
for u in urls:
    print(' ', u)

## Icechunk concepts

Think of icechunk like a **library**:

| Object | Role |
|---|---|
| `storage` | The building — where icechunk metadata lives (here: local filesystem) |
| `RepositoryConfig` | The rules — defines how the repo behaves, including where to find virtual chunk data |
| `Repository` | The catalog — versioned, transactional index of the dataset (like a git repo for arrays) |
| `Session` | A checkout — a consistent, point-in-time view of a branch (e.g. `"main"`) |
| `session.store` | The open book — the zarr-compatible interface that xarray reads from |

**What "virtual" means:** The icechunk repo stores *references* to chunks in the original S3 files — no data is copied. The `VirtualChunkContainer` tells icechunk how to fetch chunks on demand.

In [ ]:
# Local icechunk repo — data stays on S3, only references are stored locally
LOCAL_REPO = '/tmp/ndvi-icechunk'

import shutil
shutil.rmtree(LOCAL_REPO, ignore_errors=True)  # start fresh each run

storage = icechunk.local_filesystem_storage(LOCAL_REPO)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f's3://{BUCKET}/',
        store=icechunk.s3_store(region='us-east-1', anonymous=True),
    )
)

# obstore registry for virtualizarr to read the source files
store_obj = S3Store(BUCKET, region='us-east-1', skip_signature=True)
registry = ObjectStoreRegistry({f's3://{BUCKET}': store_obj})
parser = HDFParser()

## Step 1: Create virtual references

In [ ]:
%%time
vds_list = [
    open_virtual_dataset(url, parser=parser, registry=registry,
                         loadable_variables=['time', 'latitude', 'longitude'])
    for url in urls
]
print(f'Virtualized {len(vds_list)} files')
vds_list[0]

## Step 2: Concatenate along time

In [ ]:
vds = xr.concat(vds_list, dim='time', coords='minimal',
                compat='override', combine_attrs='override', data_vars='all')
vds

## Step 3: Write to local icechunk repo

In [ ]:
repo = icechunk.Repository.open_or_create(storage, config)
session = repo.writable_session('main')

vds.virtualize.to_icechunk(session.store)

msg = f'Initialize with {N_FILES} NDVI time steps'
session.commit(msg)
print(f'Committed: {msg}')

## Step 4: Read back and verify

In [ ]:
credentials = icechunk.containers_credentials(
    {f's3://{BUCKET}/': icechunk.s3_credentials(anonymous=True)}
)

read_repo = icechunk.Repository.open(storage, config,
                                     authorize_virtual_chunk_access=credentials)
read_session = read_repo.readonly_session('main')

ds = xr.open_zarr(read_session.store, consolidated=False,
                   zarr_format=3, chunks={})
ds

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np

ndvi = ds['NDVI'][0].values.astype(float)
ndvi[ndvi < -0.2] = np.nan  # mask fill values

fig, ax = plt.subplots(
    figsize=(12, 6),
    subplot_kw={'projection': ccrs.Robinson()}
)
ax.set_global()
ax.coastlines(linewidth=0.5, color='k')

im = ax.imshow(
    ndvi, origin='upper', vmin=-0.2, vmax=1.0, cmap='YlGn',
    extent=[-180, 180, -90, 90],
    transform=ccrs.PlateCarree(),
)
plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05,
             label='NDVI', shrink=0.7)
ax.set_title(f"NDVI {str(ds.time[0].values)[:10]}")
plt.tight_layout()
plt.show()